# klygo.archive (metadata.py): Inspect and validate archives

This notebook favors runnable workflows over long reference tables. It uses only generated data inside a resettable temporary directory.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample data

In [1]:
from tempfile import TemporaryDirectory
from zipfile import BadZipFile
from klygo import archive, files

if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-archive-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-archive-oybrcbsm


In [2]:
source = files.mkdir(root / "dataset")
files.save(source / "labels.txt", ["cat", "dog"], verbose=False)
files.save(source / "config.json", {"batch": 16, "classes": 2}, verbose=False)
images = files.mkdir(source / "images")
files.save(images / "cat.txt", "image placeholder", verbose=False)

sample_archive = root / "dataset.zip"
archive.compress(source, sample_archive, verbose=False)
print("Members:", archive.list_files(sample_archive))

Members: ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


## Jump to an API

[`list_files`](#list_files) · [`iter_files`](#iter_files) · [`search`](#search) · [`get_info`](#get_info) · [`test`](#test) · [`verify`](#verify) · [`compare`](#compare) · [`detect_format`](#detect_format) · [`is_archive`](#is_archive)

<a id="list_files"></a>

## `archive.list_files`

Return the member names stored in an archive.

**Signature:** `archive.list_files(archive_path: Union[str, pathlib.Path]) -> List[str]`

Returns the complete member list. Use `iter_files()` when the archive is large and a list is unnecessary.

Each block below is runnable in order.

**Basic use**

In [3]:
members = archive.list_files(sample_archive)
print("Count:", len(members))
for member in members:
    print(member)

Count: 3
dataset/config.json
dataset/images/cat.txt
dataset/labels.txt


**Practical variation**

In [4]:
text_members = [name for name in archive.list_files(sample_archive) if name.endswith(".txt")]
print(text_members)

['dataset/images/cat.txt', 'dataset/labels.txt']


**Expected error or edge behavior**

In [5]:
try:
    archive.list_files(root / "missing.zip")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-oybrcbsm\\missing.zip'


<a id="iter_files"></a>

## `archive.iter_files`

Iterate over archive member names lazily.

**Signature:** `archive.iter_files(archive_path: Union[str, pathlib.Path]) -> Iterator[str]`

Returns a lazy iterator. File access errors can occur when the iterator is consumed.

Each block below is runnable in order.

**Basic use**

In [6]:
member_iterator = archive.iter_files(sample_archive)
print(type(member_iterator).__name__)
for member in member_iterator:
    print(member)

generator
dataset/config.json
dataset/images/cat.txt
dataset/labels.txt


**Practical variation**

In [7]:
first_member = next(archive.iter_files(sample_archive))
print("First member without building a list:", first_member)

First member without building a list: dataset/config.json


**Expected error or edge behavior**

In [8]:
missing_iterator = archive.iter_files(root / "missing.zip")
try:
    next(missing_iterator)
except FileNotFoundError as error:
    print("FileNotFoundError during iteration:", error)

FileNotFoundError during iteration: [Errno 2] No such file or directory: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-oybrcbsm\\missing.zip'


<a id="search"></a>

## `archive.search`

Search archive member names using a glob or regular expression.

**Signature:** `archive.search(archive_path: Union[str, pathlib.Path], pattern: str, regex: bool = False, case_sensitive: bool = True) -> List[str]`

Uses glob matching by default or regular expressions with `regex=True`; `case_sensitive` controls both modes.

Each block below is runnable in order.

**Basic use**

In [9]:
print("Glob:", archive.search(sample_archive, "*.json"))
print("Nested glob:", archive.search(sample_archive, "*/images/*"))

Glob: ['dataset/config.json']
Nested glob: ['dataset/images/cat.txt']


**Practical variation**

In [10]:
print("Regex:", archive.search(sample_archive, r".*(labels|cat)\.txt$", regex=True))
print("Case-insensitive:", archive.search(sample_archive, "*.JSON", case_sensitive=False))

Regex: ['dataset/images/cat.txt', 'dataset/labels.txt']
Case-insensitive: ['dataset/config.json']


**Expected error or edge behavior**

In [11]:
print("No match returns an empty list:", archive.search(sample_archive, "*.csv"))

No match returns an empty list: []


<a id="get_info"></a>

## `archive.get_info`

Return detailed archive metadata and compression statistics.

**Signature:** `archive.get_info(archive_path: Union[str, pathlib.Path]) -> Dict[str, Any]`

Returns backend metadata such as format, archive size, member count, compression information, and encryption state when available.

Each block below is runnable in order.

**Basic use**

In [12]:
info = archive.get_info(sample_archive)
for key, value in info.items():
    print(f"{key}: {value}")

path: C:\Users\Mine\AppData\Local\Temp\klygo-archive-oybrcbsm\dataset.zip
format: zip
compression_algorithm: DEFLATED
encrypted: False
comment: 
file_count: 3
directory_count: 0
uncompressed_size: 67
human_uncompressed_size: 67.00 B
compressed_size: 65
human_compressed_size: 65.00 B
compress_ratio: 2.99
archive_size: 433
human_archive_size: 433.00 B
largest_file: dataset/config.json
smallest_file: dataset/labels.txt


**Practical variation**

In [13]:
print("Format:", info["format"])
print("Members:", info["file_count"])
print("Archive bytes:", info["archive_size"])

Format: zip
Members: 3
Archive bytes: 433


**Expected error or edge behavior**

In [14]:
try:
    archive.get_info(root / "missing.zip")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-oybrcbsm\\missing.zip'


<a id="test"></a>

## `archive.test`

Test archive integrity.

**Signature:** `archive.test(archive_path: Union[str, pathlib.Path], raise_exception: bool = False) -> bool`

Returns a boolean by default. With `raise_exception=True`, the backend error is propagated for diagnostics.

Each block below is runnable in order.

**Basic use**

In [15]:
print("Valid archive:", archive.test(sample_archive))

Valid archive: True


**Practical variation**

In [16]:
corrupt_source = root / "corrupted.txt"
corrupted = root / "corrupted.zip"
files.save(corrupt_source, "not a ZIP archive", verbose=False)
files.copy(corrupt_source, corrupted)
print("Corrupted archive:", archive.test(corrupted))

Corrupted archive: False


**Expected error or edge behavior**

In [17]:
try:
    archive.test(corrupted, raise_exception=True)
except (OSError, ValueError) as error:
    print(type(error).__name__, error)

ValueError ZIP archive is corrupted: File is not a zip file


<a id="verify"></a>

## `archive.verify`

Build a high-level archive verification report.

**Signature:** `archive.verify(archive_path: Union[str, pathlib.Path]) -> Dict[str, Any]`

Builds a non-raising validation report suitable for logging or status displays.

Each block below is runnable in order.

**Basic use**

In [18]:
report = archive.verify(sample_archive)
print(report)

{'valid': True, 'format': 'zip', 'file_count': 3, 'archive_size': 433, 'human_archive_size': '433.00 B', 'encrypted': False}


**Practical variation**

In [19]:
try:
    archive.verify(corrupted)
except BadZipFile as error:
    print("BadZipFile:", error)

BadZipFile: File is not a zip file


**Expected error or edge behavior**

In [20]:
try:
    archive.verify(root / "missing.zip")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-oybrcbsm\\missing.zip'


<a id="compare"></a>

## `archive.compare`

Compare the member-name sets of two archives.

**Signature:** `archive.compare(archive1: Union[str, pathlib.Path], archive2: Union[str, pathlib.Path]) -> Dict[str, List[str]]`

Compares member names, not member bytes. The result groups added, removed, and common names.

Each block below is runnable in order.

**Basic use**

In [21]:
second_source = files.mkdir(root / "second")
files.save(second_source / "labels.txt", "updated", verbose=False)
files.save(second_source / "new.txt", "new", verbose=False)
second_archive = root / "second.zip"
archive.compress(second_source, second_archive, verbose=False)
print(archive.compare(sample_archive, second_archive))

{'added_files': ['second/labels.txt', 'second/new.txt'], 'removed_files': ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt'], 'common_files': []}


**Practical variation**

In [22]:
same_copy = root / "same.zip"
archive.copy(sample_archive, same_copy)
print(archive.compare(sample_archive, same_copy))

{'added_files': [], 'removed_files': [], 'common_files': ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']}


**Expected error or edge behavior**

In [23]:
try:
    archive.compare(sample_archive, root / "missing.zip")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-archive-oybrcbsm\\missing.zip'


<a id="detect_format"></a>

## `archive.detect_format`

Detect an archive format from its name and magic bytes.

**Signature:** `archive.detect_format(path: Union[str, pathlib.Path]) -> str`

Checks known compound suffixes first and uses magic bytes for existing files before falling back to the final extension.

Each block below is runnable in order.

**Basic use**

In [24]:
for name in ("dataset.zip", "dataset.tar.gz", "dataset.tgz", "weights.gz"):
    print(name, "->", archive.detect_format(name))

dataset.zip -> zip
dataset.tar.gz -> tar.gz
dataset.tgz -> tar.gz
weights.gz -> gz


**Practical variation**

In [25]:
renamed = root / "archive.bin"
files.copy(sample_archive, renamed)
print("Detected from magic bytes:", archive.detect_format(renamed))

Detected from magic bytes: zip


**Expected error or edge behavior**

In [26]:
try:
    archive.detect_format("notes.txt")
except ValueError as error:
    print("ValueError:", error)

ValueError: Unsupported or unrecognized archive format for file 'notes.txt'. Supported formats: ['7z', 'gz', 'rar', 'tar', 'tar.bz2', 'tar.gz', 'tar.xz', 'zip']


<a id="is_archive"></a>

## `archive.is_archive`

Check whether a path appears to be a supported archive.

**Signature:** `archive.is_archive(path: Union[str, pathlib.Path]) -> bool`

A convenience predicate that converts missing, unreadable, and unsupported inputs to `False`.

Each block below is runnable in order.

**Basic use**

In [27]:
for candidate in (sample_archive, root / "missing.zip", root / "notes.txt"):
    print(candidate.name, archive.is_archive(candidate))

dataset.zip True
missing.zip True
notes.txt False


**Practical variation**

In [28]:
unknown_extension = root / "unknown.data"
files.copy(sample_archive, unknown_extension)
print("Magic bytes still work:", archive.is_archive(unknown_extension))

Magic bytes still work: True


**Expected error or edge behavior**

In [29]:
print("Unsupported input is a non-raising False:", archive.is_archive(root / "plain.txt"))

Unsupported input is a non-raising False: False


## Put it together

In [30]:
if archive.is_archive(sample_archive) and archive.test(sample_archive):
    summary = archive.get_info(sample_archive)
    matches = archive.search(sample_archive, "*.json")
    print(f"Format: {summary['format']} | Files: {summary['file_count']} | JSON matches: {matches}")

Format: zip | Files: 3 | JSON matches: ['dataset/config.json']


## Remember

- Use `is_archive()` and `test()` to validate untrusted files before extraction, and `iter_files()` when streaming large member lists.
- `detect_format()` inspects magic bytes first for existing files and falls back to filename extensions for planned output paths.
- Next in `klygo.archive`: continue with [Document_Archive_Operations.ipynb](Document_Archive_Operations.ipynb) ([Colab](https://colab.research.google.com/drive/1SwMmp98rbj2-Pyg86QWdXOkI3QO3vBxd?usp=sharing)) or return to [Document_Archive_Overview.ipynb](Document_Archive_Overview.ipynb) ([Colab](https://colab.research.google.com/drive/1z5cuH35jFOkTtSkS4IcufF8ZygnylEJS?usp=sharing)).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).